In [156]:
import LowLevelFEM as FEM
using LowLevelFEM

gmsh.initialize()

In [157]:
gmsh.open("buckling.geo")

Info    : Reading 'buckling.geo'...
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Line)
Info    : [ 20%] Meshing curve 3 (Line)
Info    : [ 30%] Meshing curve 4 (Line)
Info    : [ 40%] Meshing curve 5 (Line)
Info    : [ 50%] Meshing curve 6 (Line)
Info    : [ 60%] Meshing curve 7 (Line)
Info    : [ 60%] Meshing curve 8 (Line)
Info    : [ 70%] Meshing curve 9 (Line)
Info    : [ 80%] Meshing curve 10 (Line)
Info    : [ 90%] Meshing curve 11 (Line)
Info    : [100%] Meshing curve 12 (Line)
Info    : Done meshing 1D (Wall 0.000863299s, CPU 0.000864s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Transfinite)
Info    : [ 20%] Meshing surface 2 (Transfinite)
Info    : [ 40%] Meshing surface 3 (Transfinite)
Info    : [ 60%] Meshing surface 4 (Transfinite)
Info    : [ 70%] Meshing surface 5 (Transfinite)
Info    : [ 90%] Meshing surface 6 (Transfinite)
Info    : Done meshing 2D (Wall 0.000472164s, CPU 0.00047s)
Info    : Meshing 

In [158]:
mat = FEM.material("body", E=2.e5, ν=0.3)
problem = FEM.Problem([mat], type="Solid", bandwidth="none")

Info    : RCMK renumbering...
Info    : Done RCMK renumbering (bandwidth is now 37)


LowLevelFEM.Problem("bending3D", "Solid", 3, 3, LowLevelFEM.Material[LowLevelFEM.Material("body", 200000.0, 0.3, 7.85e-9, 45.0, 4.2e8, 1.2e-5)], 1.0, 496)

In [159]:
l = gmsh.parser.getNumber("l")[1]
a = gmsh.parser.getNumber("a")[1]
F = -100.0

-100.0

In [160]:
supp = FEM.displacementConstraint("supp", ux=0, uy=0, uz=0)
supp2 = FEM.displacementConstraint("load", ux=0, uy=0, uz=0)
load = FEM.load("load", fx=F / a^2)

("load", -1.0, 0, 0)

In [161]:
B = FEM.solveBuckling(problem, [load], [supp])

LowLevelFEM.Eigen([435.897487370538, 435.8974873706402, 3703.538437415891, 3703.538437416445, 9282.654462526682, 9282.654462527953], [-3.924420408339962e-17 -2.7755575615628914e-17 … -2.1829170833467748e-17 7.666877990472981e-18; 1.8084176349658566e-18 1.734723475976807e-18 … 2.8240621999134537e-18 8.133644609438238e-20; … ; 0.07198151758880042 0.0848813795027709 … -0.02306708384372231 0.02678394046299117; -0.049623969856857214 -0.020955084341375756 … -0.01889826907609861 -0.012909884209653702])

In [162]:
B.f

6-element Vector{Float64}:
  435.897487370538
  435.8974873706402
 3703.538437415891
 3703.538437416445
 9282.654462526682
 9282.654462527953

In [163]:
bk = FEM.showBucklingResults(problem, B, visible=true)

0

In [164]:
M = FEM.solveModalAnalysis(problem, load=[load], bc=[supp, supp2], fₘᵢₙ=1.01)

LowLevelFEM.Eigen([5093.59190974705, 5093.591911098008, 13066.843685952817, 13066.84368726736, 13611.033115698172, 23627.865559176644], [1.3350388960939253e-19 -8.398807370486872e-18 … 6.938893912999927e-18 4.253177399722788e-21; -1.3860011319403297e-17 -1.0080245379301864e-18 … -2.7755575614596553e-17 4.81739808762242e-20; … ; -0.0014200429225731859 0.00033275213141763027 … 0.0020086595686913413 -0.008379444986811467; -0.0005024943953847947 -0.0013557526545906814 … -0.0020086595602608584 -0.00419934629937723])

In [165]:
ma = FEM.showModalResults(problem, M)

1

In [166]:
gmsh.view.option.setNumber(bk, "DisplacementFactor", 300)
gmsh.view.option.setNumber(ma, "DisplacementFactor", 300)

In [167]:
E = problem.material[1].E
A = a * a
I = a^4 / 12
i = √(I / A)
β = 2
l0 = β * l
λ = l0 / i
display("λ = $λ")
σₖᵣ = π^2 * E / λ^2
Fₖᵣ = σₖᵣ * A
display("theory: Fₖᵣ = $Fₖᵣ")

Fkrfem = -F * B.f[1]
display("fem: Fₖᵣ = $Fkrfem")

"λ = 69.28203230275508"

"theory: Fₖᵣ = 41123.351671205666"

"fem: Fₖᵣ = 43589.7487370538"

In [168]:
i = 1
ρ = problem.material[1].ρ
ω0i = i^2 * π^2 / l^2 * √(E * I / (ρ * A))
ωi = i^2 * π^2 / l^2 * √(E * I / (ρ * A)) * √(1 - l^2 * F / (i^2 * π^2 * E * I))
display("i = $i")
display("ω0i = $ω0i")
display("ωi = $ωi")
display(M.f)

"i = 1"

"ω0i = 14381.00680620415"

"ωi = 14385.37744394251"

6-element Vector{Float64}:
  5093.59190974705
  5093.591911098008
 13066.843685952817
 13066.84368726736
 13611.033115698172
 23627.865559176644

In [169]:
gmsh.fltk.run()

-------------------------------------------------------
Version       : 4.13.1
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib
FLTK version  : 1.3.8
OCC version   : 7.7.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


XRequest.18: BadValue 0x0


In [170]:
gmsh.finalize()